Extract state_dicts from Lightning checkpoints, fill in each model's
card from model-cards/, and push weights + card to HF Hub.

In [13]:
import re
from pathlib import Path
import os

import torch
from huggingface_hub import ModelCard, create_repo, upload_file

In [14]:
Path(os.getcwd()).resolve().parent

PosixPath('/home/zelluzy/Desktop/code/flowers')

In [15]:
CHECKPOINTS = Path(os.getcwd()).resolve().parent / "models_checkpoints"
MODEL_CARDS = Path(os.getcwd()).resolve().parent / "model-cards"
MODEL_WEIGHTS = Path(os.getcwd()).resolve().parent / "model-weights"
MODEL_WEIGHTS.mkdir(exist_ok=True)

In [16]:
MODELS = [
    {
        "ckpt": "vit_b_16-epoch=18-val_acc=1.000-6899e8e2.ckpt",
        "card": "vit-flower-classifier-readme.md",
        "repo_id": "vit-flower-classifier",
        # Optimizer, LR scheduler, Head LR (before), Head LR (after),
        # Backbone LR (after), Unfreeze epoch, Max epochs, Batch size,
        # Effective batch size, Grad accumulation, Precision, Weight decay,
        # Early stopping patience
        "hyperparams": [
            "AdamW",
            "Cosine annealing (T_max=50, eta_min=1e-06)",
            "1e-3",
            "1e-3",
            "1e-5",
            "5",
            "50",
            "64",
            "256",
            "4",
            "16-mixed",
            "0.01",
            "5",
        ],
    },
    {
        "ckpt": "efficientnet_v2_s-epoch=25-val_acc=1.000-d6508546.ckpt",
        "card": "efficientnetv2-s-flower-classifier-readme.md",
        "repo_id": "efficientnetv2-s-flower-classifier",
        "hyperparams": [
            "AdamW",
            "Cosine annealing (T_max=30, eta_min=1e-06)",
            "1e-3",
            "1e-3",
            "1e-5",
            "5",
            "30",
            "32",
            "32",
            "1",
            "16-mixed",
            "0.01",
            "5",
        ],
    },
]

In [17]:
def extract_state_dict(ckpt_path: Path) -> dict[str, torch.Tensor]:
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    return {
        k.removeprefix("model."): v
        for k, v in ckpt["state_dict"].items()
        if k.startswith("model.")
    }

In [18]:
from huggingface_hub import HfApi

api = HfApi()

In [19]:
from safetensors.torch import save_file, load_file

for model in MODELS:
    ckpt_path = CHECKPOINTS / model["ckpt"]
    state_dict = extract_state_dict(ckpt_path)

    weight_path = MODEL_WEIGHTS / f"{model['repo_id']}.pth"
    safetensor_path = MODEL_WEIGHTS / f"{model['repo_id']}.safetensors"

    save_file(state_dict, safetensor_path)

    torch.save(state_dict, weight_path)
    print(f"Saved {weight_path} ({weight_path.stat().st_size / 1e6:.1f} MB)")

    card_path = MODEL_CARDS / model["card"]
    card = card_path.read_text()

    repo_id = f"bengid/{model['repo_id']}"

    api.create_repo(repo_id, exist_ok=True)

    api.upload_file(
        path_or_fileobj=safetensor_path,
        path_in_repo=f"{model['repo_id']}.safetensors",
        repo_id=repo_id,
    )

    card = ModelCard(content=card)
    card.push_to_hub(repo_id)

Saved /home/zelluzy/Desktop/code/flowers/model-weights/vit-flower-classifier.pth (343.6 MB)


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Saved /home/zelluzy/Desktop/code/flowers/model-weights/efficientnetv2-s-flower-classifier.pth (82.1 MB)


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.
